# Decoding a Language Model

A large language model (LLM) doesn't "answer" directly: for a given text, it computes a probability for every *token* (piece of a word) that could come next. How the next token is chosen, the **decoding**, changes the result a lot.

We use a small open model here, [Qwen3](https://huggingface.co/Qwen/Qwen3-1.7B-Base) (Apache 2.0 license). A GPU is recommended (`Runtime > Change runtime type`). Without a GPU, the notebook loads the 0.6B version of the model instead: lighter, but its texts are of lower quality.

In [1]:
import warnings

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SIZE = "1.7B" if device == "cuda" else "0.6B"  # 0.6B: light enough for a CPU

if device == "cpu":
    warnings.warn("No GPU found: the notebook loads the 0.6B model, whose texts are of lower quality. "
                  "For the 1.7B model, pick a GPU (Runtime > Change runtime type), then rerun the notebook.")

BASE_MODEL = f"Qwen/Qwen3-{MODEL_SIZE}-Base"
CHAT_MODEL = f"Qwen/Qwen3-{MODEL_SIZE}"
DTYPE = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map=device)
print(device, MODEL_SIZE)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

cuda 1.7B


## Tokens, Not Words

The model doesn't see words but *tokens*: pieces of words from a fixed vocabulary. Below, each token is shown between quotes: the space before a word is part of the token.

In [2]:
SENTENCE = "Artificial intelligence is transforming companies."

token_ids = tokenizer(SENTENCE).input_ids
print([tokenizer.decode(token_id) for token_id in token_ids])
print(len(SENTENCE.split()), "words,", len(token_ids), "tokens")

['Art', 'ificial', ' intelligence', ' is', ' transforming', ' companies', '.']
5 words, 7 tokens


## The Probability of the Next Token

Given the beginning of a sentence, the model gives a probability to each of the ~150,000 tokens of its vocabulary. Here are the 10 most likely ones:

In [3]:
CAPITAL_PROMPT = "The Eiffel Tower is located in"

def next_token_table(prompt: str, k: int = 10) -> pd.DataFrame:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probabilities = torch.softmax(logits.float(), dim=-1)
    top = torch.topk(probabilities, k)
    return pd.DataFrame({
        "token": [repr(tokenizer.decode(i)) for i in top.indices],
        "probability": top.values.cpu().numpy().round(3),
    })


next_token_table(CAPITAL_PROMPT)

,token,probability
0,' Paris',0.234
1,' the',0.206
2,' which',0.156
3,' __',0.069
4,' ____',0.060
5,' what',0.033
6,' France',0.028
7,' (',0.022
8,'\n',0.020
9,' a',0.020


The model learned these probabilities by reading huge amounts of text: it has no base of facts, only statistical regularities.

## Greedy Decoding

The simplest strategy: take the most likely token at each step, append it to the text, and repeat.

In [4]:
STORY_PROMPT = "The best way to learn a foreign language is"

def greedy(prompt: str, n_tokens: int = 80) -> str:
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        next_id = logits.argmax()
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tokenizer.decode(ids[0], skip_special_tokens=True)


print(greedy(STORY_PROMPT))

The best way to learn a foreign language is to practice speaking with native speakers. This is the best way to learn a foreign language. The best way to learn a foreign language is to practice speaking with native speakers. This is the best way to learn a foreign language. The best way to learn a foreign language is to practice speaking with native speakers. This is the best way to learn a foreign language. The best way to learn a foreign language


Greedy decoding is deterministic: the same beginning always gives the same continuation, often bland. It also tends to repeat itself, even to loop: look at the end of the text.

## Sampling and Temperature

Rather than taking the most likely token, we can **draw** the next token according to the model's probabilities. The **temperature** sets the boldness: low, we stick to the most likely tokens; high, we allow unlikely choices.

In [5]:
def generate(prompt: str, **kwargs) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id,
                                **kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [6]:
for temperature in [0.2, 0.8, 1.5]:
    set_seed(0)
    print(f"--- temperature = {temperature}")
    print(generate(STORY_PROMPT, do_sample=True, temperature=temperature, top_k=0))

--- temperature = 0.2
The best way to learn a foreign language is to speak it. The more you speak, the better you will become. You can improve your language skills by talking to people in the language you want to learn. You can also improve your language skills
--- temperature = 0.8
The best way to learn a foreign language is to _____ language in daily life. [ ]
A. speak
B. listen to
C. read
D. write
Answer: A

—What is across the road, Jack?—
--- temperature = 1.5
The best way to learn a foreign language is actually hung from fire Put gapboard 根据情景最适合填什么词。nullptr 看起来这则变俄的说法不通。

看來口径不足以分辨 across Spain菲罢市中心 Livingston得


At low temperature the text is safe; at high temperature it becomes creative, then incoherent.

## Top-k and Top-p: Cutting Unlikely Choices

To avoid unlikely choices, we can draw only among the most likely tokens:

- *top-k* keeps the `k` most likely tokens;
- *top-p* (or *nucleus sampling*) keeps the most likely ones whose cumulative probability reaches `p`.

We keep variety without derailing, and many assistants combine these settings with the temperature. Three draws of each, three different texts:

In [7]:
settings = {
    "top-k = 10": dict(top_k=10),
    "top-p = 0.9": dict(top_k=0, top_p=0.9),  # top_k=0: only the top-p acts (else transformers applies top_k=50)
}
for name, kwargs in settings.items():
    for seed in range(3):
        set_seed(seed)
        print(f"--- {name}, seed = {seed}")
        print(generate(STORY_PROMPT, do_sample=True, temperature=0.8, **kwargs))

--- top-k = 10, seed = 0
The best way to learn a foreign language is to use it. The best way to learn a foreign language is to use it. You can use it in your daily life. You can use it to write letters to friends. You can use it
--- top-k = 10, seed = 1
The best way to learn a foreign language is to speak it, but it is not easy to learn to speak a foreign language. You have to speak the language as a native does. If you have never lived in another country, you will have
--- top-k = 10, seed = 2
The best way to learn a foreign language is to practice listening and speaking with native speakers, as well as reading and writing. You can also use language learning apps to practice and improve your skills. Additionally, watching movies or TV shows in the target
--- top-p = 0.9, seed = 0
The best way to learn a foreign language is to __________.
A. pay attention to what you hear
B. pay attention to what you say
C. pay attention to what you read
D. pay attention to what you write

--- top-p = 

The texts stay readable, but not always on topic: a small base model readily moves on to a fill-in-the-blank exercise or a translation, both common in its training data (hence also the `__` among the Eiffel Tower's likely tokens).

## Base Model or "Assistant" Model?

A base model only learned to **continue** text. Let's ask it a question:

In [8]:
QUESTION = "What are three benefits of remote work for a company?"

set_seed(0)
print(generate(QUESTION, do_sample=True, temperature=0.7, top_p=0.9))

What are three benefits of remote work for a company? Additionally, could you explain what is meant by "digital transformation" and provide three examples of how companies can achieve it? How can a company use artificial intelligence to improve its services? What is a "


It continues the text: sometimes it answers, sometimes it moves on to other questions or drifts, since nothing pushes it to answer. Assistants (ChatGPT, Claude, Le Chat, …) are base models **fine-tuned** to follow instructions, then aligned with human preferences. Same question to the "assistant" version of the same model:

In [9]:
del model  # free the memory of the base model
if device == "cuda":
    torch.cuda.empty_cache()

chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, dtype=DTYPE, device_map=device)

messages = [{"role": "user", "content": QUESTION}]
inputs = chat_tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                            return_dict=True, return_tensors="pt").to(device)
set_seed(0)
with torch.no_grad():
    output = chat_model.generate(**inputs, max_new_tokens=400, do_sample=True,
                                 temperature=0.7, top_p=0.9)
prompt_length = inputs["input_ids"].shape[1]
print(chat_tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Three benefits of remote work for a company include:

1. **Increased Flexibility and Productivity**: Remote work allows employees to work from anywhere, which can lead to better work-life balance and higher job satisfaction. This flexibility often results in increased productivity and efficiency.

2. **Cost Savings**: Companies can reduce overhead costs such as office space, utilities, and supplies by eliminating the need for a physical office. These savings can be reinvested into other areas of the business.

3. **Wider Talent Pool**: Remote work enables companies to hire talent from a broader geographic area, which can help attract and retain top talent that might not be located in the company's original area.


## A Reasoning Model

Recent assistants can **reason** before answering: they first write a draft, between `<think>` and `</think>` tags, then their answer. This draft is generated text like any other, token by token: the user often doesn't see it, but it is computed, and billed. Qwen3 can do both: above, we turned this mode off (`enable_thinking=False`). Same question, reasoning on, with the decoding settings Qwen recommends for this mode:

In [10]:
inputs = chat_tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=True,
                                            return_dict=True, return_tensors="pt").to(device)
set_seed(0)
with torch.no_grad():
    output = chat_model.generate(**inputs, max_new_tokens=1500, do_sample=True,
                                 temperature=0.6, top_p=0.95)
generated = output[0, inputs["input_ids"].shape[1]:].tolist()
end_of_thinking = chat_tokenizer.convert_tokens_to_ids("</think>")
# without </think>, max_new_tokens cut the reasoning short, before the answer
split = generated.index(end_of_thinking) + 1 if end_of_thinking in generated else len(generated)
print(f"--- reasoning ({split} tokens)")
print(chat_tokenizer.decode(generated[:split], skip_special_tokens=True).strip())
print(f"\n--- answer ({len(generated) - split} tokens)")
print(chat_tokenizer.decode(generated[split:], skip_special_tokens=True).strip())

--- reasoning (238 tokens)
<think>
Okay, the user is asking for three benefits of remote work for a company. Let me think about this. First, I need to recall what I know about remote work. Remote work allows employees to work from anywhere, right? So, one benefit could be cost savings for the company. They don't have to pay for office space, utilities, or equipment. That's a solid point.

Another thought: increased flexibility. Employees can manage their time better, which might lead to higher productivity. Also, companies can hire talent from a broader pool, not just local areas. That's another benefit.

Wait, maybe there's another one. Oh, remote work can reduce overhead costs. The company saves on office space, utilities, and maybe even benefits like health insurance. That's a third point. Let me check if these are accurate. Yes, cost savings are a common benefit. Flexibility in work hours and location can improve employee satisfaction and retention. And broader talent pool is defin

The draft comes on top of the answer: compare the two token counts printed. That is compute time and, through an API, money, for a text the user often doesn't read. A small model may also switch language along the way, to Chinese for Qwen, the other main language of its training data: look at the end of the answer. Reasoning helps mostly with multi-step tasks (math, code, planning); on a question this simple, it adds little.

## Takeaways

- An LLM computes probabilities of next tokens; the text depends on how we choose among them
- Temperature, top-k and top-p set the trade-off between reliability and creativity
- The same prompt can give different answers: keep it in mind when testing and evaluating
- An assistant is a base model fine-tuned to follow instructions
- A reasoning model writes a draft before its answer: more reliable on multi-step problems, but more tokens, so slower and more expensive